# MixGRPO 最小可运行 Demo

本 notebook 对应 `12_Mix_grpo.ipynb`，展示 MixGRPO 的核心：

- 总去噪链有 `T` 步；
- 当前滑动窗口 `W(l)` 内使用随机 SDE，形成可计算 log-prob 的短 MDP；
- 窗口外使用确定性 ODE，不计算也不伪造 log-prob；
- PPO/GRPO loss 只 replay 窗口内 `W` 步；
- 窗口按训练迭代逐步从早期去噪阶段滑向后期。

参考：MixGRPO 论文、`开源模型参考/RL/MixGRPO/fastvideo/train_grpo_flux.py`、`fastvideo/utils/grpo_states.py`、`sampling_utils.py`，以及当前项目 `online_RL` 的 `MixGRPO`/`WindowScheduler`。


## 1. 混合链的数据契约

本例采用 $t:0\to1$ 的去噪方向。对第 $k$ 步：

$$x_{k+1}=\begin{cases}x_k+\Delta t\,v_{old}(x_k,t_k)+\psi\sqrt{\Delta t}\epsilon_k,&k\in W(l)\\x_k+\Delta t\,v_{old}(x_k,t_k),&k\notin W(l).\end{cases}$$

保存 `states [G,T+1,D]`，但仅保存 `sde_actions/old_logp [G,W]`。同一个条件组中的 G 个候选共享初始噪声；在进入 SDE 窗口前它们完全相同，候选差异只由窗口内随机动作产生。

训练目标为窗口内 PPO surrogate：

$$L=-\frac1{GW}\sum_{g,k\in W(l)}\min(r_{g,k}A_g,\operatorname{clip}(r_{g,k},1-\epsilon,1+\epsilon)A_g).$$


In [ ]:
import math
from copy import deepcopy

import torch
import torch.nn as nn

torch.manual_seed(37)
G, T, D = 16, 8, 2
WINDOW = 2
ITERS_PER_WINDOW = 2
NOISE = 0.32
CLIP_RANGE = 1e-3  # MixGRPO 常用非常紧的 ratio clip
LR = 3e-3

class TinyFlowPolicy(nn.Module):
    def __init__(self, hidden=48):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(D + 2, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, D),
        )
    def forward(self, x, t, cond):
        return self.net(torch.cat([x, t[:, None], cond], dim=-1))

def reward_fn(x, cond):
    goal = torch.cat([1.4 + cond, -0.7 + 0.3 * cond], dim=-1)
    return -(x - goal).square().sum(-1)

def group_advantages(reward, eps=1e-6):
    return ((reward - reward.mean()) / (reward.std(correction=0) + eps)).detach()


In [ ]:
class SlidingWindow:
    """最小 progressive scheduler；到末尾后回到第 0 步。"""
    def __init__(self, total_steps, width, iters_per_window, stride=1):
        self.total_steps = total_steps
        self.width = width
        self.iters_per_window = iters_per_window
        self.stride = stride
        self.start = 0
        self.iteration = 0

    def active_steps(self):
        return list(range(self.start, min(self.start + self.width, self.total_steps)))

    def step(self):
        self.iteration += 1
        if self.iteration % self.iters_per_window == 0:
            self.start += self.stride
            if self.start > self.total_steps - self.width:
                self.start = 0

scheduler = SlidingWindow(T, WINDOW, ITERS_PER_WINDOW, stride=1)
for i in range(10):
    print(f'iteration={i:02d}, window={scheduler.active_steps()}')
    scheduler.step()
# 重新初始化，供正式 demo 使用。
scheduler = SlidingWindow(T, WINDOW, ITERS_PER_WINDOW, stride=1)


In [ ]:
@torch.no_grad()
def mixed_rollout(old, cond, active_steps):
    """窗口内 SDE、窗口外 ODE；返回窗口动作的 frozen anchor。"""
    dt = 1.0 / T
    active_set = set(active_steps)
    # 同组共享初始噪声：[1,D] -> [G,D]。
    initial = torch.randn(1, D)
    x = initial.expand(cond.shape[0], -1).clone()
    states = [x.clone()]
    old_logps, old_means, stds = [], [], []
    sde_step_indices = []

    for step in range(T):
        t = torch.full((x.shape[0],), step / T)
        mean = x + dt * old(x, t, cond)
        if step in active_set:
            std = torch.full_like(mean, NOISE * math.sqrt(dt))
            dist = torch.distributions.Normal(mean, std)
            x_next = dist.sample()
            old_logps.append(dist.log_prob(x_next).sum(-1))
            old_means.append(mean)
            stds.append(std)
            sde_step_indices.append(step)
        else:
            x_next = mean  # 纯 ODE：没有随机动作，也没有 log-prob。
        states.append(x_next.clone())
        x = x_next

    return {
        'states': torch.stack(states, 1),
        'old_logp': torch.stack(old_logps, 1),
        'old_means': torch.stack(old_means, 1),
        'stds': torch.stack(stds, 1),
        'sde_indices': sde_step_indices,
    }

def replay_window(theta, rollout, cond):
    """只 replay rollout 当时的 SDE 窗口，返回 [G,W] log-prob。"""
    dt = 1.0 / T
    new_logps, new_means = [], []
    for local_idx, step in enumerate(rollout['sde_indices']):
        x = rollout['states'][:, step]
        action = rollout['states'][:, step + 1].detach()
        t = torch.full((x.shape[0],), step / T)
        mean = x + dt * theta(x, t, cond)
        dist = torch.distributions.Normal(mean, rollout['stds'][:, local_idx])
        new_logps.append(dist.log_prob(action).sum(-1))
        new_means.append(mean)
    return torch.stack(new_logps, 1), torch.stack(new_means, 1)


In [ ]:
def mix_grpo_loss(new_logp, old_logp, advantages, clip_range=CLIP_RANGE):
    """advantages=[G]，扩展到窗口 [G,W]；只对 W 步归约。"""
    adv = advantages[:, None].expand_as(new_logp)
    ratio = (new_logp - old_logp).exp()
    surr1 = ratio * adv
    surr2 = ratio.clamp(1 - clip_range, 1 + clip_range) * adv
    loss = -torch.minimum(surr1, surr2).mean()
    clip_fraction = ((ratio.detach() - 1).abs() > clip_range).float().mean()
    return loss, ratio, clip_fraction


In [ ]:
theta = TinyFlowPolicy()
optimizer = torch.optim.Adam(theta.parameters(), lr=LR)
cond = torch.zeros(G, 1)

for iteration in range(12):
    active = scheduler.active_steps()
    old = deepcopy(theta).eval()
    for p in old.parameters():
        p.requires_grad_(False)

    rollout = mixed_rollout(old, cond, active)
    rewards = reward_fn(rollout['states'][:, -1], cond)
    advantages = group_advantages(rewards)
    new_logp, new_means = replay_window(theta, rollout, cond)
    loss, ratio, clip_fraction = mix_grpo_loss(
        new_logp, rollout['old_logp'].detach(), advantages
    )

    optimizer.zero_grad()
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(theta.parameters(), 1.0)
    optimizer.step()

    # 窗口推进必须发生在本轮 optimizer step 之后。
    scheduler.step()
    diversity_before = rollout['states'][:, active[0]].std(dim=0).mean().item()
    diversity_after = rollout['states'][:, active[-1] + 1].std(dim=0).mean().item()
    print(f'iter={iteration:02d} window={active} reward={rewards.mean().item():+.3f} grad={grad_norm.item():.3f} '
          f'clip={clip_fraction.item():.2f} diversity={diversity_before:.3f}->{diversity_after:.3f}')

print('full states:', tuple(rollout['states'].shape))
print('trainable log-probs:', tuple(new_logp.shape), 'active indices:', rollout['sde_indices'])
assert rollout['states'].shape == (G, T + 1, D)
assert new_logp.shape == rollout['old_logp'].shape == (G, WINDOW)
assert len(rollout['sde_indices']) == WINDOW
assert torch.isfinite(loss)


## 2. 正确实现检查表

- mixed sampling 与 sliding-window loss 必须同时实现；只在 loss 中切片、rollout 仍全程 SDE，并不是完整 MixGRPO；
- 同一 prompt/场景的候选必须使用同一个窗口位置，不能每个候选随机选窗口；
- ODE 步没有随机转移密度，不应填零 log-prob 后混入 ratio；
- replay 必须使用 rollout 当时保存的 `sde_indices`，不能使用已经滑动后的新窗口；
- 窗口应在 optimizer step 后更新；
- MixGRPO-Flash 的高阶 ODE 加速是进一步优化，本最小 demo 未实现；它不改变窗口内 SDE/PPO 的核心目标。

与当前项目对应：`WindowScheduler.get_sde_mask()` 决定 rollout 哪些步随机，`MixGRPO.compute_loss()` 只选择相同窗口的 log-prob。若框架只调用后者而没有把 mask 传入采样器，就只实现了“窗口 loss”，尚未完整实现“Mixed ODE-SDE rollout”。
